In [ ]:
# =============================================================================
# Oxford (Progress Tracker) -> dataset de la tabla avance_oxford
# El archivo viene ancho: dos filas de encabezado y un par Completed/Score por bloque.
# =============================================================================
import re
from pathlib import Path

import pandas as pd

ORIGEN = Path(r"C:\Users\sebastian.mendez\Downloads\Libro1.xlsx")  # archivo o carpeta
QUITAR_PROMEDIO = True          # la fila "Class Average" no es un estudiante


def texto(serie):
    """Serie de texto limpio y sin NaN, para comparar encabezados sin sorpresas."""
    return serie.astype("string").fillna("").str.strip()


def a_porcentaje(serie):
    """'42.9%' -> 42.9   |   '-', 'Never', vacío -> NaN"""
    return pd.to_numeric(texto(serie).str.replace("%", "", regex=False),
                         errors="coerce").round(2)


def a_fecha(serie):
    """'03 Dec 2025' -> 2025-12-03   |   'Never' y '-' -> NaT"""
    return pd.to_datetime(texto(serie), format="%d %b %Y", errors="coerce")


def partir_bloque(titulo):
    """'Unit 1 Achievements' -> (1, 'Achievements', 'Practice')
       'Unit 1 Test'         -> (1, 'Test', 'Test')
       'Total'               -> (NA, 'Total / General', 'Overall')"""
    titulo = " ".join(str(titulo).split())
    if titulo.lower().startswith("total"):
        return pd.NA, "Total / General", "Overall"
    partes = re.match(r"(?i)^unit\s+(\d+)\s*(.*)$", titulo)
    if not partes:
        return pd.NA, titulo, "Practice"
    resto = partes[2].strip() or "Unit"
    return int(partes[1]), resto, ("Test" if resto.lower() == "test" else "Practice")


def a_largo(crudo, origen=""):
    """Convierte una hoja ancha del Progress Tracker en filas (una por bloque y estudiante)."""
    # 1. La fila que dice "Completed" marca donde termina el encabezado
    es_subtitulo = crudo.apply(lambda fila: texto(fila).str.lower().eq("completed").any(), axis=1)
    if not es_subtitulo.any():
        return None                       # esta hoja no es del Progress Tracker
    fila_sub = es_subtitulo.idxmax()
    if fila_sub == 0:
        return None                       # no hay fila de titulos arriba

    subtitulos = texto(crudo.loc[fila_sub])            # Teacher, Student, Completed, Score...
    # 2. El nombre del bloque solo esta encima de "Completed"; lo de al lado se arrastra
    titulos = (texto(crudo.loc[fila_sub - 1])
               .replace(r"^_\d+$", pd.NA, regex=True)  # Excel pone _1, _2... en las vacias
               .replace("", pd.NA).ffill().fillna(""))
    datos = crudo.loc[fila_sub + 1:].reset_index(drop=True)

    def columna(nombre):
        """Numero de la columna cuyo subtitulo es ese (o None si no esta)."""
        halladas = subtitulos.index[subtitulos.str.lower() == nombre]
        return halladas[0] if len(halladas) else None

    c_docente, c_material = columna("teacher"), columna("learning material")
    c_estudiante, c_fecha = columna("student"), columna("date")

    # 3. Docente y material vienen solo en la fila de "Class Average": se arrastran hacia abajo
    fijas = pd.DataFrame({
        "teacher": texto(datos[c_docente]).replace("", pd.NA).ffill() if c_docente is not None else pd.NA,
        "learning_material": texto(datos[c_material]).replace("", pd.NA).ffill() if c_material is not None else pd.NA,
        "student": texto(datos[c_estudiante]) if c_estudiante is not None else pd.NA,
        "last_opened": a_fecha(datos[c_fecha]) if c_fecha is not None else pd.NaT,
    })

    # 4. Un pedazo por cada par Completed/Score
    piezas = []
    for orden, inicio in enumerate(subtitulos.index[subtitulos.str.lower() == "completed"]):
        numero, unidad, tipo = partir_bloque(titulos[inicio])
        siguiente = inicio + 1
        hay_score = siguiente in subtitulos.index and subtitulos[siguiente].lower() == "score"
        piezas.append(fijas.assign(
            unit_number=numero,
            unit_title=unidad,
            item_type=tipo,
            completed_pct=a_porcentaje(datos[inicio]),
            score_pct=a_porcentaje(datos[siguiente]) if hay_score else pd.NA,
            _fila=range(len(datos)),      # para dejar juntas las filas de cada estudiante
            _orden=orden,
        ))

    largo = pd.concat(piezas, ignore_index=True)
    largo = largo.sort_values(["_fila", "_orden"]).drop(columns=["_fila", "_orden"])
    largo = largo[largo["student"].ne("")]
    if QUITAR_PROMEDIO:
        largo = largo[largo["student"].str.lower() != "class average"]
    largo["unit_number"] = largo["unit_number"].astype("Int64")
    largo["origen"] = origen              # de que archivo/hoja salio (se quita al final)
    return largo.reset_index(drop=True)


def leer_oxford(ruta):
    """Lee un archivo (.xlsx con varias hojas o .csv) y devuelve las filas del Progress Tracker."""
    ruta = Path(ruta)
    partes = []
    if ruta.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        for hoja in pd.ExcelFile(ruta).sheet_names:
            crudo = pd.read_excel(ruta, sheet_name=hoja, header=None, dtype=object)
            largo = a_largo(crudo, f"{ruta.name} | {hoja}") if len(crudo) else None
            print(f"  {ruta.name} | {hoja}: " + (f"{len(largo)} filas" if largo is not None
                                                 else "no es del Progress Tracker, se salta"))
            if largo is not None:
                partes.append(largo)
    else:
        crudo = pd.read_csv(ruta, header=None, dtype=object, encoding="utf-8-sig",
                            sep=None, engine="python")
        largo = a_largo(crudo, ruta.name)
        print(f"  {ruta.name}: " + (f"{len(largo)} filas" if largo is not None
                                    else "no es del Progress Tracker, se salta"))
        if largo is not None:
            partes.append(largo)
    return partes


archivos = sorted(ORIGEN.glob("*.*")) if ORIGEN.is_dir() else [ORIGEN]
partes = [pieza for archivo in archivos for pieza in leer_oxford(archivo)]

avance_oxford = pd.concat(partes, ignore_index=True)
avance_oxford["created_at"] = pd.Timestamp.now()
avance_oxford = avance_oxford[["teacher", "learning_material", "student", "last_opened",
                               "unit_number", "unit_title", "item_type",
                               "completed_pct", "score_pct", "created_at"]]

print(f"\nDataset listo: {len(avance_oxford):,} filas x {len(avance_oxford.columns)} columnas")
avance_oxford.head(30)

Hello Word


In [ ]:
# --- Cronometro: en que paso se demora ---
import time

print("ORIGEN:", ORIGEN, "| ¿es carpeta?", ORIGEN.is_dir(), "| ¿existe?", ORIGEN.exists())

t = time.time(); hojas = pd.ExcelFile(ORIGEN).sheet_names
print(f"abrir el libro: {time.time()-t:.1f}s  -> {hojas}")

for hoja in hojas:
    t = time.time(); crudo = pd.read_excel(ORIGEN, sheet_name=hoja, header=None, dtype=object)
    print(f"  leer '{hoja}': {time.time()-t:.1f}s  -> {crudo.shape}")
    if len(crudo):
        t = time.time(); largo = a_largo(crudo, hoja)
        print(f"  transformar '{hoja}': {time.time()-t:.1f}s  -> "
              + (f"{len(largo)} filas" if largo is not None else "se salta"))